# synth_data_gen.ipynb

Sinh du lieu NER tong hop (BANGIAO §3-§6). Chay tren Kaggle GPU T4 x2, Internet ON.

**Output:** `ner_synth_data.zip` chua `input/*.txt` + `label/*.json`

Logic nam trong `fakeer/src/synth_pipeline.py` -- notebook chi goi.

In [ ]:
import os, sys, subprocess
os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True)

REPO = 'https://github.com/Khanhhh239/fakeer.git'
if not os.path.exists('fakeer'):
    subprocess.run(['git', 'clone', '-q', REPO], check=True)
subprocess.run(['git', '-C', 'fakeer', 'pull', '-q', 'origin', 'main'])

sys.path.insert(0, 'fakeer/src')
import json, random, time

# Thu muc output
OUT_DIR   = '/kaggle/working/ner_synth'
TXT_DIR   = f'{OUT_DIR}/input'
JSON_DIR  = f'{OUT_DIR}/label'
ZIP_PATH  = '/kaggle/working/ner_synth_data.zip'
for d in [TXT_DIR, JSON_DIR]: os.makedirs(d, exist_ok=True)

# So file muon sinh (doi thanh 500 khi chay chinh thuc)
N_TARGET = 10  # test: 10 file

print('Setup xong')

In [ ]:
import torch, inspect
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

_ngpu = max(1, torch.cuda.device_count())
print(f'{_ngpu} GPU kha dung')

def _mk(model, tp, util):
    want = dict(model=model, dtype='float16',
                gpu_memory_utilization=util, tensor_parallel_size=tp,
                enforce_eager=True, disable_custom_all_reduce=True)
    ok = set(inspect.signature(LLM.__init__).parameters)
    return LLM(**{k: v for k, v in want.items() if k in ok})

LLM_MODEL = 'Qwen/Qwen3-8B'
_plans = [(LLM_MODEL, _ngpu, 0.90), (LLM_MODEL, _ngpu, 0.80),
          ('Qwen/Qwen2.5-7B-Instruct', _ngpu, 0.85)]
llm = None
for _m, _tp, _u in _plans:
    try:
        print(f'Thu {_m} | TP={_tp} | util={_u}')
        llm = _mk(_m, _tp, _u); LLM_MODEL = _m; break
    except Exception as e:
        print('  x', type(e).__name__, str(e)[:120])
assert llm is not None, 'Khong nap duoc LLM nao'
qtok = AutoTokenizer.from_pretrained(LLM_MODEL)
print('DUNG:', LLM_MODEL)

In [ ]:
from synth_pipeline import build_kb, sample_diagnoses, build_t1_prompts, parse_t1_batch

KB_DIR = 'fakeer/kb'
kb = build_kb(KB_DIR)
print(f'Chan doan: {len(kb["chandoan_pool"])}, Trieu chung: {len(kb["trieuchung_pool"])}')
print(f'Thuoc: {len(kb["thuoc_pool"])}, XN: {len(kb["xn_names"])}')
print(f'Am thuoc: {len(kb["am_thuoc"])}, Am XN: {len(kb["am_xetnghiem"])}')

# Phan tang: n_per_chapter=1 cho test 10 file, doi thanh 5 khi chay 500
n_per_chapter = max(1, N_TARGET // 20)
sample_diag = sample_diagnoses(kb['chandoan_pool'], n_per_chapter=n_per_chapter)
print(f'Chan doan mau: {len(sample_diag)} (n_per_chapter={n_per_chapter})')

t0 = time.time()
t1_outs = llm.generate(build_t1_prompts(sample_diag, qtok),
                       SamplingParams(temperature=0.3, max_tokens=300))
scenarios = parse_t1_batch(t1_outs, sample_diag, kb['thuoc_norm_set'])
print(f'T1: {len(t1_outs)} kich ban / {time.time()-t0:.0f}s -> {len(scenarios)} hop le')
print('Vi du:', scenarios[0] if scenarios else None)

In [ ]:
from synth_pipeline import build_t2b_inputs, process_t2b_outputs

t2b_inputs = build_t2b_inputs(scenarios, kb, qtok)
t2b_prompts = [x[0] for x in t2b_inputs]

t0 = time.time()
t2b_outs = llm.generate(t2b_prompts, SamplingParams(temperature=0.7, max_tokens=1100))
print(f'T2B: {len(t2b_outs)} bai / {time.time()-t0:.0f}s')

saved_t2b, reject_log = process_t2b_outputs(
    t2b_outs, t2b_inputs, kb, llm, qtok, SamplingParams,
    TXT_DIR, JSON_DIR, saved_start=1
)
total_gen = saved_t2b + sum(reject_log.values())
pct = sum(reject_log.values()) / max(1, total_gen) * 100
print(f'T2B luu: {saved_t2b}, bo qua: {sum(reject_log.values())} ({pct:.0f}%)')
print('Ly do bo qua:', reject_log)
if pct > 40: print('!!! > 40% bi loai -- xem lai prompt')

In [ ]:
from synth_pipeline import process_t2a_blocks
from synth_struct import build_blocks
from synth_source import gen_lab_pairs

def make_struct_entities(scenarios):
    result = []
    for sc in scenarios:
        s = sc['scenario']
        ents = [(sc['diagnosis']['term'], 'CHAN_DOAN')]
        for t in s['tc']: ents.append((t, 'TRIEU_CHUNG'))
        for t in s['th']: ents.append((t, 'THUOC'))
        for t in s['tx']: ents.append((t, 'TEN_XET_NGHIEM'))
        for full, ten, kq in gen_lab_pairs(3):
            ents.append((ten, 'TEN_XET_NGHIEM'))
            if kq: ents.append((kq, 'KET_QUA_XET_NGHIEM'))
        result.append(ents)
    return result

n_struct = max(2, int(N_TARGET * 0.35))
struct_ents_list = make_struct_entities(scenarios)
blocks = build_blocks(struct_ents_list, n_struct, kb['heading_ls'], kb['heading_hd'])
saved_t2a = process_t2a_blocks(blocks, TXT_DIR, JSON_DIR, saved_start=saved_t2b+1)
print(f'T2A luu: {saved_t2a} khoi cau truc')
print(f'Tong file: {saved_t2b + saved_t2a}')

In [ ]:
from collections import Counter
from synth_pipeline import zip_output

all_ents = []
for fn in sorted(os.listdir(JSON_DIR)):
    if not fn.endswith('.json'): continue
    all_ents.extend(json.load(open(f'{JSON_DIR}/{fn}', encoding='utf-8')))

cnt = Counter(e['type'] for e in all_ents)
total = len(all_ents)
n_files = len([f for f in os.listdir(JSON_DIR) if f.endswith('.json')])
print(f'Tong: {total} thuc the trong {n_files} file')
TARGET = {'CHAN_DOAN': 25, 'TRIEU_CHUNG': 33, 'THUOC': 16,
          'TEN_XET_NGHIEM': 15, 'KET_QUA_XET_NGHIEM': 11}
for t, tgt in sorted(TARGET.items(), key=lambda x: -x[1]):
    n = cnt.get(t, 0)
    pct2 = n / max(1, total) * 100
    flag = 'ok' if abs(pct2 - tgt) <= 15 else 'LECH'
    print(f'  {flag} {t}: {n} ({pct2:.1f}%, muc tieu {tgt}%)')

zip_output(OUT_DIR, ZIP_PATH)
print(f'\nTai ve: {ZIP_PATH}')